# DAG-SA EEG — Colab orchestration (run from GitHub)

Annealed search over DAG ensembles for subject-specific motor-imagery EEG decoding.

**Open this notebook in Colab straight from GitHub:** in Colab, *File -> Open notebook -> GitHub*, paste your repo URL, and pick `DAG_SA_Colab.ipynb`.
Then run the cells top to bottom: clone -> install -> mount Drive -> smoke test -> full runs.

Code lives in GitHub; the (large) datasets and all results live on Google Drive, so nothing big is ever committed.


## 0. Clone the repo
Set `REPO_URL` to your repository. For a private repo, use a token URL:
`https://<TOKEN>@github.com/yazanjer/DAG-Ensembles-EEG.git`.


In [ ]:
REPO_URL = 'https://github.com/yazanjer/DAG-Ensembles-EEG.git'   # for a private repo: https://<TOKEN>@github.com/yazanjer/DAG-Ensembles-EEG.git
import os, sys, subprocess, glob
CLONE_DIR = '/content/eeg_dagsa_repo'
if not os.path.exists(CLONE_DIR):
    subprocess.run(['git','clone','--depth','1',REPO_URL,CLONE_DIR], check=True)
else:
    subprocess.run(['git','-C',CLONE_DIR,'pull'])
# locate the folder containing env_utils.py (repo root or an EEG/ subfolder)
hits = glob.glob(CLONE_DIR+'/**/env_utils.py', recursive=True)
assert hits, 'env_utils.py not found in the cloned repo'
CODE_DIR = os.path.dirname(hits[0])
sys.path.insert(0, CODE_DIR); os.chdir(CODE_DIR)
print('code dir:', CODE_DIR)


## 1. Install dependencies
`torch` is pre-installed on Colab GPU runtimes; installed here only if missing.


In [ ]:
pkgs=['numpy','scipy','scikit-learn','pandas','matplotlib','networkx','pyyaml','mne','pyriemann']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs])
try:
    import torch
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','torch'])
print('deps ready')


## 2. Mount Drive and set PROJECT_ROOT
Datasets and results live on Drive. Upload your data to Drive so Colab can read it:

```
MyDrive/EEG_DAGSA/dataset/BCICIV_calib_ds1a.mat ...   (Dataset 1)
MyDrive/EEG_DAGSA/dataset_2a/A01T.mat ...             (Dataset 2a)
```


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/EEG_DAGSA'   # data + results live here
import env_utils
paths, cfg = env_utils.setup_environment(project_root=PROJECT_ROOT)
print('Put Dataset 1 .mat files in:', paths.dataset_dir)


## 3. Smoke test (must be green before full runs)


In [ ]:
import run_experiments, datasets_io
subj = datasets_io.available_subjects('ds1', paths.dataset_dir)
print('ds1 subjects found:', subj)
summary, exp_dir = run_experiments.run_experiment(
    dataset='ds1', subjects=subj[:1], seeds=[42,43],
    protocol='split', experiment='smoke', tiny=True)
summary


## 4. Full multi-seed — Dataset 1 (all 8 seeds, R2-3)


In [ ]:
summary_ds1, dir_ds1 = run_experiments.run_experiment(
    dataset='ds1', subjects=['a','b','f','g'], seeds=cfg['seeds'],
    protocol='split', experiment='ds1_multiseed')
summary_ds1


### 4b. Cross-validation protocol (R1-2)


In [ ]:
summary_cv, dir_cv = run_experiments.run_experiment(
    dataset='ds1', subjects=['a','b'], seeds=cfg['seeds'][:3],
    protocol='cv', experiment='ds1_cv')
summary_cv


## 5. Dataset 2a — 9 subjects (R1-3)
Upload the 2a `A0?T.mat` files to `MyDrive/EEG_DAGSA/dataset_2a/` and point `dataset_dir` there.


In [ ]:
DS2A_DIR = str(paths.root / 'dataset_2a')   # edit if you used another Drive path
subj2a = datasets_io.available_subjects('ds2a', DS2A_DIR)
print('ds2a subjects:', subj2a)
summary_2a, dir_2a = run_experiments.run_experiment(
    dataset='ds2a', subjects=subj2a, seeds=cfg['seeds'],
    protocol='split', experiment='ds2a_multiseed',
    dataset_dir=DS2A_DIR, variant='binary')
summary_2a


## 6. Ablations (R2-8, R2-9)


In [ ]:
import ablations
df_abl, freq = ablations.run_ablations(dataset='ds1', subjects=['a'],
    seeds=cfg['seeds'][:2], tiny=False)
freq


## 7. Outputs
All results are written to `MyDrive/EEG_DAGSA/results/<experiment>/`:
`baseline_comparison_multiseed.csv`, `significance_summary.csv`, `winloss_summary.csv`,
`exp2b_baseline_comparison.pdf`, `confusion_*.pdf`, `convergence_*.pdf`, `config_used.yaml`,
`leakage_audit.txt`, `search_space.txt`. Checkpoints under `checkpoints/` resume after a disconnect.
